In [0]:
#Example 1: Creating and Inserting Data into a Delta Table
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("DeltaTableExamples") \
    .getOrCreate()

# Create a sample DataFrame
data = [("Alice", 34), ("Bob", 45), ("Catherine", 29)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)

# Write DataFrame to Delta Table
#df.write.format("delta").mode("overwrite").save("/delta/people")community edition will not support
df.write.format("delta").mode("overwrite").saveAsTable("people")

# Verify the Delta Table content
#delta_df = spark.read.format("delta").load("/delta/people") community edition will not support
delta_df = spark.read.table("people")
delta_df.show()

In [0]:
#Example 2: Upsert Data Using MERGE INTO (Delta Lake Merge)
from delta.tables import DeltaTable

# Create an existing Delta Table instance
#delta_table = DeltaTable.forPath(spark, "/delta/people") community Edition will not support
delta_table = DeltaTable.forName(spark, "people")

# New data to upsert
new_data = [("Alice", 35), ("David", 28)]
new_df = spark.createDataFrame(new_data, ["name", "age"])

# Create a temporary view from the new DataFrame
new_df.createOrReplaceTempView("new_data")

# Perform upsert using MERGE INTO
delta_table.alias("target").merge(
    source = spark.table("new_data").alias("source"),
    condition = "target.name = source.name"
).whenMatchedUpdate(set={"age": "source.age"}) \
 .whenNotMatchedInsert(values={"name":"source.name", "age":"source.age"})\
 .execute()

# Verify the changes
delta_table.toDF().show()

In [0]:
#Example 3: Implementing Time Travel in Delta Tables
# Display current version of Delta Table
#current_df = spark.read.format("delta").load("/delta/people") because DBFS is disable for community edition
current_df = spark.read.table("people")
current_df.show()

# Query a previous version of the Delta Table (e.g., version 0)
#version_0_df = spark.read.format("delta").option("versionAsOf", 1).load("/delta/people")
version_0_df = spark.read.format("delta").option("versionAsOf", 0).table("people")
version_0_df.show()

# Query using timestamp (if known)
# timestamp_df = spark.read.format("delta").option("timestampAsOf", "2024-11-12T01:00:00.000Z").load("/delta/people")
# timestamp_df.show()

